# 머신러닝을 위한 선형대수학
## 06 고윳값, 고유벡터, 행렬의 대각화

---
**목차**
1. 고윳값, 고유벡터의 정의
2. 행렬의 대각화
3. 대칭 행렬의 대각화 (Eigen Decomposition)
4. 양의 정부호 행렬 (Positive Definite Matrix)
5. 외적과 행렬의 제곱근
6. 스펙트럴 클러스터링 (Spectral Clustering)

In [ ]:
# ── 라이브러리 로드 ───

# ── 라이브러리 임포트 ───
# NumPy — 행렬·벡터 연산, 선형대수 핵심 라이브러리
import numpy as np
# Matplotlib — 그래프·벡터 시각화
import matplotlib.pyplot as plt
# patches — 도형·화살표 등 그래픽 요소
import matplotlib.patches as mpatches
# patches — 도형·화살표 등 그래픽 요소
from matplotlib.patches import FancyArrowPatch
# SciPy 선형대수 — sqrtm 등
from scipy.linalg import eig, sqrtm
# scikit-learn — 데이터·클러스터링
from sklearn.datasets import make_moons, make_circles
# scikit-learn — 데이터·클러스터링
from sklearn.cluster import SpectralClustering, KMeans

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

print('라이브러리 로드 완료!')


---
## 1. 고윳값, 고유벡터의 정의

### 📌 정의 6.1
하나의 $n \times n$ 행렬 $A : \mathbb{R}^n \to \mathbb{R}^n$ 을 선형변환이라 하자.  
**0이 아닌** 벡터 $v \in \mathbb{R}^n$ 가 다음 조건을 만족할 때:

$$Av = \lambda v$$

- $\lambda$: 행렬 $A$의 **고윳값(eigenvalue)**
- $v$: 고윳값 $\lambda$에 대한 **고유벡터(eigenvector)**

### 💡 직관적 이해
고유벡터는 행렬 $A$에 의해 변환되어도 **방향이 변하지 않고**, 크기만 $\lambda$배 변하는 특별한 벡터입니다.

In [ ]:
# ── 강의 예제 Av = λv 검증 ───

# 강의 예제: A = [[1, -8], [1, -5]]
A = np.array([[1, -8], [1, -5]])
# 고유벡터 v₁, λ₁=-3
v1 = np.array([2, 1])  # 고유벡터 1
v2 = np.array([4, 1])  # 고유벡터 2

# 구분선 출력 — 섹션 구분
print('=' * 50)
print('행렬 A:')
print(A)
print()
print('고유벡터 v1 =', v1)
print('Av1 =', A @ v1)
print('-3 * v1 =', -3 * v1)
print('=> Av1 = -3 * v1  (고윳값 λ₁ = -3)')
print()
print('고유벡터 v2 =', v2)
print('Av2 =', A @ v2)
print('-1 * v2 =', -1 * v2)
print('=> Av2 = -1 * v2  (고윳값 λ₂ = -1)')


In [ ]:
# ── 고유벡터 방향 불변 시각화 ───

# 시각화: 고유벡터는 선형변환 후 방향이 유지됨
# Matplotlib — 개념을 그림으로 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#e74c3c', '#3498db']
vectors = [v1, v2]
eigenvalues = [-3, -1]
labels = ['v₁ = (2,1), λ₁ = -3', 'v₂ = (4,1), λ₂ = -1']

for idx, ax in enumerate(axes):
    v = vectors[idx]
    lam = eigenvalues[idx]
# Av = λv — 방향만 유지, 크기 λ배
    Av = A @ v
    
    # 정규화해서 시각화
    scale = 1.0
    
    # 원래 고유벡터
    ax.annotate('', xy=v * scale, xytext=[0, 0],
                arrowprops=dict(arrowstyle='->', color=colors[idx], lw=2.5))
    ax.text(v[0] * scale + 0.1, v[1] * scale + 0.1,
            f'v{idx+1}', fontsize=13, color=colors[idx], fontweight='bold')
    
    # 변환 후 벡터 Av = λv
    ax.annotate('', xy=Av * 0.5, xytext=[0, 0],
                arrowprops=dict(arrowstyle='->', color='gray', lw=2.5, linestyle='dashed'))
    ax.text(Av[0] * 0.5 + 0.1, Av[1] * 0.5 + 0.1,
            f'Av{idx+1} = λ{idx+1}·v{idx+1}', fontsize=11, color='gray')
    
    ax.set_xlim(-4, 5)
    ax.set_ylim(-3, 3)
    ax.axhline(y=0, color='k', linewidth=0.5)
    ax.axvline(x=0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)
    ax.set_title(f'{labels[idx]}\n방향 유지, 크기만 변화', fontsize=12, fontweight='bold')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    
    # 원점 표시
    ax.plot(0, 0, 'ko', markersize=5)

# 전체 figure 제목
plt.suptitle('고유벡터는 선형변환 후 방향(선)이 변하지 않는다!', fontsize=14, fontweight='bold', y=1.02)
# 서브플롯 간격 자동 조정
plt.tight_layout()
# 파일로 저장 (선택)
plt.savefig('/tmp/eigenvector_viz.png', bbox_inches='tight', dpi=100)
# 그래프 화면 출력
plt.show()
print('핵심: 고유벡터 v에 A를 곱해도 같은 방향(선) 위에 있다!')


---
## 고윳값 구하기: 특성다항식

### 📌 정의 6.2 (특성다항식)
$n \times n$ 행렬 $A$에 대하여:
- **특성다항식**: $f(\lambda) = \det(\lambda I - A)$
- **특성방정식**: $f(\lambda) = \det(\lambda I - A) = 0$

### 🔑 고윳값 구하는 절차
1. 특성방정식 $\det(\lambda I - A) = 0$ 을 풀어 **고윳값 $\lambda$** 구하기
2. 각 $\lambda$에 대해 $(\lambda I - A)v = 0$ 을 풀어 **고유벡터 $v$** 구하기

In [ ]:
# ── 예제1 고윳값·고유벡터 계산 ───

# 예제 1: A = [[1, -8], [1, -5]] 의 고윳값, 고유벡터 계산
print('예제 1: A = [[1, -8], [1, -5]]')
# 구분선 출력 — 섹션 구분
print('=' * 50)
A = np.array([[1., -8.], [1., -5.]])
print('행렬 A:')
print(A)
print()

# numpy로 고윳값, 고유벡터 계산
# numpy eig로 Av=λv 해
# 고윳값·고유벡터 계산 (Av = λv)
eigenvalues, eigenvectors = np.linalg.eig(A)
print('고윳값:', eigenvalues)
print('고유벡터 (열벡터로 표현):')
print(eigenvectors)
print()

# 손으로 계산한 것과 비교
print('손 계산 결과: λ₁ = -3, λ₂ = -1')
print('특성방정식: λ² + 4λ + 3 = 0  =>  (λ+3)(λ+1) = 0')
print()

# 검증
for i, lam in enumerate(eigenvalues):
    v = eigenvectors[:, i]
    Av = A @ v
    lv = lam * v
    print(f'λ{i+1} = {lam:.1f}: Av = {Av.round(4)}, λv = {lv.round(4)}, 일치: {np.allclose(Av, lv)}')


In [ ]:
# ── 예제2 중근 λ=2 ───

# 예제 2: A = [[3, 1], [-1, 1]] - 중근이 나오는 경우
print('예제 2: A = [[3, 1], [-1, 1]] (중근 λ = 2)')
# 구분선 출력 — 섹션 구분
print('=' * 50)
A2 = np.array([[3., 1.], [-1., 1.]])
# 고윳값·고유벡터 계산 (Av = λv)
eigenvalues2, eigenvectors2 = np.linalg.eig(A2)
print('고윳값:', eigenvalues2)
print('특성방정식: (λ-2)² = 0  => λ = 2 (중근)')
print('고유벡터: v = c(1, -1)ᵀ')
print()
print('참고: 고윳값은 2개이지만, 여기에 대응하는 고유벡터는 무수히 많다!')
print('(c ≠ 0인 임의의 상수에 대해 c(1, -1)ᵀ이 모두 고유벡터)')


In [ ]:
# ── 예제3 3×3 행렬 ───

# 예제 3: 3x3 행렬
print('예제 3: 3×3 행렬 A = [[-2,0,0],[-4,1,-1],[4,0,2]]')
# 구분선 출력 — 섹션 구분
print('=' * 50)
A3 = np.array([[-2., 0., 0.], [-4., 1., -1.], [4., 0., 2.]])
print('행렬 A:')
print(A3)
print()

# 고윳값·고유벡터 계산 (Av = λv)
eigenvalues3, eigenvectors3 = np.linalg.eig(A3)
print('고윳값:', np.sort(eigenvalues3.real))
print('=> λ = -2, 1, 2')
print('특성방정식: (λ+2)(λ-1)(λ-2) = 0')
print()
# 반복: 각 원소/조합에 대해 계산·검증
for i in range(3):
    print(f'λ{i+1} = {eigenvalues3[i].real:.1f}: 고유벡터 ≈ {eigenvectors3[:,i].real.round(3)}')


In [ ]:
# ── 특성다항식 f(λ) 그래프 ───

# 시각화: 특성다항식 f(λ) = det(λI - A) 그래프
# 2×2 특성다항식 det(λI-A)
# 함수 char_poly_2x2: 알고리즘·목적 설명은 docstring 참고
def char_poly_2x2(A, lam):
    """2x2 행렬의 특성다항식 계산"""
    return (lam - A[0,0]) * (lam - A[1,1]) - A[0,1] * A[1,0]

A_ex = np.array([[1., -8.], [1., -5.]])
lam_range = np.linspace(-5, 2, 500)
f_lam = [char_poly_2x2(A_ex, l) for l in lam_range]

# Matplotlib — 개념을 그림으로 시각화
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(lam_range, f_lam, 'b-', linewidth=2, label='f(λ) = det(λI - A) = λ² + 4λ + 3')
ax.axhline(y=0, color='k', linewidth=1)
ax.axvline(x=0, color='k', linewidth=0.5, alpha=0.5)

# 고윳값 표시
roots = [-3, -1]
for r in roots:
    ax.axvline(x=r, color='red', linestyle='--', alpha=0.7)
    ax.plot(r, 0, 'ro', markersize=10, zorder=5)
    ax.annotate(f'λ = {r}\n(고윳값)', xy=(r, 0), xytext=(r+0.3, 8),
                fontsize=11, color='red',
                arrowprops=dict(arrowstyle='->', color='red'))

ax.set_xlabel('λ', fontsize=13)
ax.set_ylabel('f(λ)', fontsize=13)
ax.set_title('특성다항식 f(λ) = det(λI - A)\n고윳값 = 특성방정식 f(λ) = 0의 해', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_ylim(-5, 20)
# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()


In [ ]:
# ── A^100 고유값 분해 활용 ───

# A^100 계산 시연 (고유값 분해 활용)
print('🚀 고유값 분해로 A^100 쉽게 계산하기!')
# 구분선 출력 — 섹션 구분
print('=' * 50)
A = np.array([[1., -8.], [1., -5.]])
P = np.array([[2., 4.], [1., 1.]])
D = np.array([[-3., 0.], [0., -1.]])

print('A = P * D * P⁻¹ 에서')
print('A^100 = P * D^100 * P⁻¹')
print()
print('D^100 =')
D100 = np.array([[(-3)**100, 0], [0, (-1)**100]])
print(f'[[(-3)^100, 0], [0, (-1)^100]] = [[3^100, 0], [0, 1]]')
print()

# 검증 (작은 거듭제곱으로)
A5_direct = np.linalg.matrix_power(A, 5)
# 역행렬 계산
P_inv = np.linalg.inv(P)
# D^k 대角 — 거듭제곱 쉬움
D5 = np.diag([(-3)**5, (-1)**5])
A5_decomp = P @ D5 @ P_inv

print('검증: A^5 직접 계산 vs 고유값 분해 계산')
print('직접 계산:')
print(A5_direct)
print('고유값 분해:')
print(A5_decomp.round(6))
print('일치:', np.allclose(A5_direct, A5_decomp))


---
## 2. 행렬의 대각화

### 📌 정의 6.3
$n \times n$ 행렬 $A$에 대하여, $n \times n$인 정칙행렬 $P$가 존재하여:
$$P^{-1}AP = D$$
를 만족하면 **$A$는 대각화 가능한 행렬**이라 부른다. ($D$는 대각행렬)

### 📌 정리 6.5 (대각화 가능 여부 판단)
$n \times n$ 행렬 $A$에 대하여, **$n$개의 고유벡터 $v_1, \ldots, v_n$이 일차독립**이면 행렬 $A$를 대각화할 수 있다.

### ⚠️ 주의
- 고윳값이 모두 다른 경우 → **항상 대각화 가능**
- 고윳값에 중근이 있는 경우 → **대각화 가능할 수도, 불가능할 수도 있음**

In [ ]:
# ── 대각화 함수 diagonalize ───

# 대각화 예제 - 단계별 설명
# A = PDP⁻¹ 대각화 단계별
# 함수 diagonalize: 알고리즘·목적 설명은 docstring 참고
def diagonalize(A, verbose=True):
    """행렬 A를 대각화하는 함수"""
# 고윳값·고유벡터 계산 (Av = λv)
    eigenvalues, eigenvectors = np.linalg.eig(A)
    
    if verbose:
        print('행렬 A:')
        print(A)
        print()
        print('Step 1: 고윳값 계산')
        print('고윳값:', np.round(eigenvalues, 4))
        print()
        print('Step 2: 고유벡터 계산 (P의 열벡터들)')
        print('P (고유벡터 행렬):')
        print(np.round(eigenvectors, 4))
        print()
    
    P = eigenvectors
    D = np.diag(eigenvalues)
# 역행렬 계산
    P_inv = np.linalg.inv(P)
    
    if verbose:
        print('Step 3: D = P⁻¹AP 검증')
        D_check = P_inv @ A @ P
        print('D = P⁻¹AP =')
        print(np.round(D_check.real, 4))
        print()
        print('대각행렬 D (고윳값들이 대각에):')
        print(np.round(D.real, 4))
        print()
        print('대각화 성공!:', np.allclose(D_check.real, D.real, atol=1e-10))
    
    return eigenvalues, P, D

# 예제 5: 고유치가 모두 다른 경우
print('예제 5: 고유치가 모두 다른 경우 - 대각화 가능')
# 구분선 출력 — 섹션 구분
print('=' * 60)
A5 = np.array([[-2., 0., 0.], [-4., 1., -1.], [4., 0., 2.]])
eigenvalues5, P5, D5 = diagonalize(A5)


In [ ]:
# ── 대각화 불가능·가능 사례 ───

# 대각화 불가능한 경우
print('예제 6: 고윳값이 중근인 경우 - 대각화 불가능')
# 구분선 출력 — 섹션 구분
print('=' * 60)
B = np.array([[3., 2., 0.], [0., 2., 0.], [1., 1., 2.]])
print('행렬 B:')
print(B)
print()
# 고윳값·고유벡터 계산 (Av = λv)
eigenvalues_B, eigenvectors_B = np.linalg.eig(B)
print('고윳값:', np.round(np.sort(eigenvalues_B.real), 4))
print('=> λ = 3, 2 (2는 중근)')
print()
print('λ = 2 (중근)인 경우: rank([λI-B]) = 2')
lam = 2
# (λI-A) rank로 고유벡터 개수 판단
M = lam * np.eye(3) - B
print('λI - B =')
print(M)
# 행렬 rank — 독립 행/열 개수
print(f'rank(λI-B) = {np.linalg.matrix_rank(M)}')
print('rank = 2 이므로 선형독립인 고유벡터를 2개 구할 수 없다 => 대각화 불가능!')
print()

# 대각화 가능한 중근 예제
print('예제 7: 고윳값이 중근이지만 대각화 가능한 경우')
# 구분선 출력 — 섹션 구분
print('=' * 60)
C = np.array([[0., -1., -1.], [1., 2., 1.], [1., 1., 2.]])
print('행렬 C:')
print(C)
# 고윳값·고유벡터 계산 (Av = λv)
eigenvalues_C, eigenvectors_C = np.linalg.eig(C)
print('고윳값:', np.round(np.sort(eigenvalues_C.real), 4))
print()
lam1 = 1
M1 = lam1 * np.eye(3) - C
# 행렬 rank — 독립 행/열 개수
print(f'λ = 1 (중근)인 경우: rank(λI-C) = {np.linalg.matrix_rank(M1)}')
print('rank = 1 이므로 선형독립인 고유벡터를 2개 구할 수 있다 => 대각화 가능!')


In [ ]:
# ── 대각화 기하학적 의미 ───

# 시각화: 대각화의 기하학적 의미
# Matplotlib — 개념을 그림으로 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

A_demo = np.array([[1., -8.], [1., -5.]])

# 단위원 위의 점들
theta = np.linspace(0, 2*np.pi, 100)
circle = np.array([np.cos(theta), np.sin(theta)])

# 고윳값·고유벡터 계산 (Av = λv)
eigenvalues_demo, P_demo = np.linalg.eig(A_demo)
P_demo = P_demo.real
eigenvalues_demo = eigenvalues_demo.real
# 역행렬 계산
P_inv_demo = np.linalg.inv(P_demo)

# 변환된 점들
# A·단위원 → 타원
transformed = A_demo @ circle

# P의 열벡터 (고유벡터)
axes[0].plot(circle[0], circle[1], 'b-', linewidth=2, label='단위원')
# 반복: 각 원소/조합에 대해 계산·검증
for i in range(2):
    v = P_demo[:, i]
    axes[0].annotate('', xy=v, xytext=[0,0],
                    arrowprops=dict(arrowstyle='->', color=['red','green'][i], lw=2.5))
    axes[0].text(v[0]+0.05, v[1]+0.05, f'v{i+1}', fontsize=12, color=['red','green'][i], fontweight='bold')
axes[0].set_xlim(-5, 5)
axes[0].set_ylim(-3, 3)
axes[0].axhline(y=0, color='k', lw=0.5)
axes[0].axvline(x=0, color='k', lw=0.5)
axes[0].set_title('원래 공간\n(고유벡터 방향)', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[0].set_aspect('equal')

axes[1].plot(transformed[0], transformed[1], 'r-', linewidth=2, label='A * 단위원')
axes[1].axhline(y=0, color='k', lw=0.5)
axes[1].axvline(x=0, color='k', lw=0.5)
axes[1].set_title('A에 의한 선형변환\n(타원으로 변환)', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 대각화의 단계
steps = ['P⁻¹', 'D (스케일)', 'P']
axes[2].text(0.5, 0.7, 'A = P D P⁻¹', fontsize=16, ha='center', fontweight='bold',
            transform=axes[2].transAxes)
axes[2].text(0.5, 0.55, '= (고유벡터 기저로 변환) × (스케일링) × (원래 기저로 복귀)',
            fontsize=10, ha='center', transform=axes[2].transAxes)
axes[2].text(0.5, 0.35, f'D = diag({eigenvalues_demo[0]:.0f}, {eigenvalues_demo[1]:.0f})',
            fontsize=14, ha='center', color='purple', transform=axes[2].transAxes)
axes[2].text(0.5, 0.2, f'고윳값: λ₁={eigenvalues_demo[0]:.0f}, λ₂={eigenvalues_demo[1]:.0f}',
            fontsize=12, ha='center', color='gray', transform=axes[2].transAxes)
axes[2].axis('off')
axes[2].set_title('대각화의 의미', fontweight='bold')

# 전체 figure 제목
plt.suptitle('행렬의 대각화: A = P D P⁻¹', fontsize=14, fontweight='bold')
# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()


---
## 3. 대칭 행렬의 대각화 (Eigen Decomposition)

### 📌 핵심 정리 6.9
$n \times n$ 행렬 $A$가 **대칭행렬** ($A^T = A$) 이면, **정규직교행렬** $Q$ ($Q^TQ = I$) 가 존재하여:
$$A = Q^T D Q \quad (D \text{는 대각행렬})$$

### 📌 정리 6.8 (대칭행렬의 직교 고유벡터)
대칭행렬에서 **서로 다른 고윳값에 대응하는 고유벡터들은 서로 직교**한다.

In [ ]:
# ── 대칭행렬 대각화 예제10 ───

# 예제 10: 대칭행렬의 대각화
print('예제 10: 대칭행렬 A = [[1,2],[2,1]] 대각화')
# 구분선 출력 — 섹션 구분
print('=' * 60)

A10 = np.array([[1., 2.], [2., 1.]])
print('A:')
print(A10)
print('A^T == A?', np.allclose(A10, A10.T), '=> 대칭행렬!')
print()

# 대칭 → 직교 고유벡터
# 고윳값·고유벡터 계산 (Av = λv)
# 대칭행렬 전용 고윳값 분해 (실수 고윳값 보장)
eigenvalues10, eigenvectors10 = np.linalg.eigh(A10)  # eigh: 대칭행렬용
print('고윳값:', eigenvalues10)
print('고유벡터 (정규화된):')
print(eigenvectors10)
print()

# 직교성 확인
v1_10 = eigenvectors10[:, 0]
v2_10 = eigenvectors10[:, 1]
dot_product = np.dot(v1_10, v2_10)
print(f'v₁ · v₂ = {dot_product:.10f} ≈ 0 => 직교!')
print()

# 정규직교행렬 Q 구성
Q = eigenvectors10
D = np.diag(eigenvalues10)
print('Q^T Q =')
print(np.round(Q.T @ Q, 6))
print('=> 단위행렬! (정규직교행렬 확인)')
print()

print('A = Q^T D Q 검증:')
A_reconstructed = Q.T @ D @ Q  # 주의: numpy eigh는 Q로 Q^TDQ = A
print(np.round(A_reconstructed, 4))
print('원래 A와 일치:', np.allclose(A_reconstructed, A10))


In [ ]:
# ── 3×3 대칭행렬 대각화 ───

# 예제 11: 3x3 대칭행렬 대각화
print('예제 11: 3×3 대칭행렬 대각화')
# 구분선 출력 — 섹션 구분
print('=' * 60)
A11 = np.array([[2., 1., 1.], [1., 2., 1.], [1., 1., 2.]])
print('A:')
print(A11)
print()

# 고윳값·고유벡터 계산 (Av = λv)
# 대칭행렬 전용 고윳값 분해 (실수 고윳값 보장)
eigenvalues11, Q11 = np.linalg.eigh(A11)
D11 = np.diag(eigenvalues11)

print('고윳값:', eigenvalues11)
print('=> λ = 1 (중근), λ = 4')
print()
print('정규직교 고유벡터 행렬 Q:')
print(np.round(Q11, 4))
print()

# 직교성 확인
print('Q^T Q (단위행렬이어야 함):')
print(np.round(Q11.T @ Q11, 6))
print()

# Eigen Decomposition 검증
A_reconstructed11 = Q11 @ D11 @ Q11.T
print('Q D Q^T = A 검증:')
print(np.round(A_reconstructed11, 4))
print('성공:', np.allclose(A_reconstructed11, A11))


In [ ]:
# ── 대칭행렬 고유벡터 직교성 ───

# 시각화: 대칭행렬의 고유벡터 직교성
# Matplotlib — 개념을 그림으로 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

A_sym = np.array([[2., 1.], [1., 2.]])
# 고윳값·고유벡터 계산 (Av = λv)
# 대칭행렬 전용 고윳값 분해 (실수 고윳값 보장)
eigenvalues_sym, Q_sym = np.linalg.eigh(A_sym)

# 왼쪽: 단위원과 고유벡터
ax1 = axes[0]
theta = np.linspace(0, 2*np.pi, 200)
circle = np.array([np.cos(theta), np.sin(theta)])
transformed_sym = A_sym @ circle

ax1.plot(circle[0], circle[1], 'b--', linewidth=1.5, alpha=0.5, label='단위원')
ax1.plot(transformed_sym[0], transformed_sym[1], 'b-', linewidth=2, label='A * 단위원')

colors_ev = ['#e74c3c', '#2ecc71']
# 반복: 각 원소/조합에 대해 계산·검증
for i in range(2):
    v = Q_sym[:, i]
    lam = eigenvalues_sym[i]
    Av = A_sym @ v
    
    # 고유벡터 그리기
    ax1.annotate('', xy=v * 1.5, xytext=[0, 0],
                arrowprops=dict(arrowstyle='->', color=colors_ev[i], lw=3))
    # 변환된 벡터 그리기
    ax1.annotate('', xy=Av * 0.5, xytext=[0, 0],
                arrowprops=dict(arrowstyle='->', color=colors_ev[i], lw=2, linestyle='dashed', alpha=0.6))
    ax1.text(v[0]*1.7, v[1]*1.7, f'v{i+1}\n(λ={lam:.0f})', fontsize=11, color=colors_ev[i], fontweight='bold', ha='center')

# 직각 표시 (직교성)
# 직각 표시
# ── 라이브러리 임포트 ───
# patches — 도형·화살표 등 그래픽 요소
from matplotlib.patches import Arc
ax1.plot([0], [0], 'ko', markersize=6)
ax1.text(-0.5, 0.2, '90°', fontsize=12, color='purple')

ax1.set_xlim(-3.5, 3.5)
ax1.set_ylim(-3, 3.5)
ax1.axhline(y=0, color='k', lw=0.5)
ax1.axvline(x=0, color='k', lw=0.5)
ax1.set_title('대칭행렬의 고유벡터는 서로 직교!\nA = [[2,1],[1,2]]', fontweight='bold', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')

# 오른쪽: 일반 행렬 vs 대칭 행렬 비교
ax2 = axes[1]
categories = ['일반 행렬', '대칭 행렬']
properties = [
    ['실수 고윳값\n(보장 안됨)', '직교 고유벡터\n(보장 안됨)', '항상 대각화\n(보장 안됨)'],
    ['실수 고윳값\n✓ 항상 보장', '직교 고유벡터\n✓ 항상 보장', '항상 대각화\n✓ 항상 가능']
]
colors_table = [['#ffcccc', '#ffcccc', '#ffcccc'],
                ['#ccffcc', '#ccffcc', '#ccffcc']]

ax2.axis('off')
table_data = [['특성', '일반 행렬', '대칭 행렬'],
              ['실수 고윳값', '보장 안됨 ✗', '항상 보장 ✓'],
              ['직교 고유벡터', '보장 안됨 ✗', '항상 보장 ✓'],
              ['대각화 가능성', '조건부 ✗', '항상 가능 ✓'],
              ['분해 형태', 'P⁻¹AP = D', 'Q^T AQ = D']]

table = ax2.table(cellText=table_data[1:], colLabels=table_data[0],
                  loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.3, 2.5)

# 색상 적용
# 반복: 각 원소/조합에 대해 계산·검증
for j in range(3):
    table[0, j].set_facecolor('#4a4a8a')
    table[0, j].set_text_props(color='white', fontweight='bold')

# 반복: 각 원소/조합에 대해 계산·검증
for i in range(1, 5):
    table[i, 1].set_facecolor('#ffe0e0')
    table[i, 2].set_facecolor('#e0ffe0')

ax2.set_title('일반 행렬 vs 대칭 행렬 비교', fontweight='bold', fontsize=13, pad=20)

# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()


---
## 4. 양의 정부호 행렬 (Positive Definite Matrix)

### 📌 정의 6.11
대칭행렬 $A$ ($A^T = A$)에 대해:

| 종류 | 조건 | 기호 |
|------|------|------|
| **양의 정부호 (PDM)** | 모든 $x \neq 0$에 대해 $x^T A x > 0$ | PDM |
| **양의 준정부호 (PSDM)** | 모든 $x$에 대해 $x^T A x \geq 0$ | PSDM |

### 📌 정리 6.13 (동치 조건)
다음은 모두 동치:
1. 행렬 $A$가 양의 정부호 행렬
2. $A$의 모든 eigenvalue가 **양의 실수**
3. 정칙 행렬 $U$가 존재하여 $A = U^T U$
4. $A$의 모든 sub-determinant가 양의 실수

In [ ]:
# ── 양의 정부호 행렬 확인 ───

# PDM: 대칭 + 모든 λ>0 + x^TAx>0
# 함수 check_positive_definite: 알고리즘·목적 설명은 docstring 참고
def check_positive_definite(A, name='A'):
    """양의 정부호 행렬 여부 확인"""
    print(f'행렬 {name}:')
    print(A)
    print()
    
    # 대칭 확인
    is_symmetric = np.allclose(A, A.T)
    print(f'1. 대칭행렬: {is_symmetric}')
    
    if not is_symmetric:
        print('=> 대칭행렬이 아니므로 PDM 분석 불가')
        return
    
    # 고윳값 확인
# 고윳값·고유벡터 계산 (Av = λv)
    eigenvalues = np.linalg.eigvalsh(A)
    all_positive = np.all(eigenvalues > 0)
    all_nonneg = np.all(eigenvalues >= 0)
    print(f'2. 고윳값: {np.round(eigenvalues, 4)}')
    print(f'   모든 고윳값 > 0: {all_positive}')
    print(f'   모든 고윳값 >= 0: {all_nonneg}')
    
    # xᵀAx > 0 확인 (여러 랜덤 벡터로)
    np.random.seed(42)
    n_tests = 1000
    min_quad = float('inf')
# 반복: 각 원소/조합에 대해 계산·검증
    for _ in range(n_tests):
        x = np.random.randn(A.shape[0])
        quad = x @ A @ x
        min_quad = min(min_quad, quad)
    print(f'3. 이차형식 x^TAx 최솟값 (랜덤 {n_tests}개): {min_quad:.6f}')
    
    if all_positive:
        print(f'\n=> {name}는 양의 정부호 행렬 (PDM) ✓')
    elif all_nonneg:
        print(f'\n=> {name}는 양의 준정부호 행렬 (PSDM) ✓')
    else:
        print(f'\n=> {name}는 PDM도 PSDM도 아님 ✗')
    return eigenvalues

# 예제들
print('예제: 양의 정부호 행렬 확인')
# 구분선 출력 — 섹션 구분
print('=' * 60)
A_pdm = np.array([[2., -1., 0.], [-1., 2., -1.], [0., -1., 2.]])
ev = check_positive_definite(A_pdm, 'A')


In [ ]:
# ── A^TA, AA^T는 PSDM ───

# PSDM: A^TA, AA^T는 항상 양의 준정부호 행렬
print('보기 14: 임의의 행렬 A에 대해 A^TA, AA^T는 양의 준정부호 행렬')
# 구분선 출력 — 섹션 구분
print('=' * 60)
A_arb = np.array([[1., 2., 3.], [4., 5., 6.]])
print('임의의 행렬 A (2×3):')
print(A_arb)
print()

# A^TA는 항상 양의 준정부호
ATA = A_arb.T @ A_arb
AAT = A_arb @ A_arb.T

print('A^TA (3×3):')
check_positive_definite(ATA, 'A^TA')
print()
print('AA^T (2×2):')
check_positive_definite(AAT, 'AA^T')


In [ ]:
# ── 이차형식 x^TAx 시각화 ───

# 시각화: 이차형식 x^TAx
# Matplotlib — 개념을 그림으로 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x_range = np.linspace(-2, 2, 100)
y_range = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x_range, y_range)

# PDM: 양의 정부호
A_pos = np.array([[3., 1.], [1., 2.]])
# 고윳값·고유벡터 계산 (Av = λv)
eigenvalues_pos = np.linalg.eigvalsh(A_pos)
# PDM → 원점에서 볼록 (양수)
Z_pos = np.array([[np.array([X[i,j], Y[i,j]]) @ A_pos @ np.array([X[i,j], Y[i,j]])
# 반복: 각 원소/조합에 대해 계산·검증
                   for j in range(100)] for i in range(100)])

c1 = axes[0].contourf(X, Y, Z_pos, levels=20, cmap='RdYlGn')
plt.colorbar(c1, ax=axes[0])
axes[0].contour(X, Y, Z_pos, levels=[0], colors='black', linewidths=2)
axes[0].set_title(f'양의 정부호 행렬 (PDM)\nx^TAx > 0 (항상 양수)\n고윳값: {eigenvalues_pos.round(2)}', fontweight='bold')
axes[0].set_xlabel('x₁')
axes[0].set_ylabel('x₂')
axes[0].set_aspect('equal')
axes[0].plot(0, 0, 'k*', markersize=12, label='원점 (최솟값)')
axes[0].legend(fontsize=10)

# 부정부호 행렬 (indefinite)
A_indef = np.array([[1., 0.], [0., -1.]])
# 고윳값·고유벡터 계산 (Av = λv)
eigenvalues_indef = np.linalg.eigvalsh(A_indef)
Z_indef = np.array([[np.array([X[i,j], Y[i,j]]) @ A_indef @ np.array([X[i,j], Y[i,j]])
# 반복: 각 원소/조합에 대해 계산·검증
                     for j in range(100)] for i in range(100)])

c2 = axes[1].contourf(X, Y, Z_indef, levels=20, cmap='RdYlGn')
plt.colorbar(c2, ax=axes[1])
axes[1].contour(X, Y, Z_indef, levels=[0], colors='black', linewidths=2)
axes[1].set_title(f'부정부호 행렬 (Indefinite)\nx^TAx는 양수도 음수도 됨\n고윳값: {eigenvalues_indef.round(2)}', fontweight='bold')
axes[1].set_xlabel('x₁')
axes[1].set_ylabel('x₂')
axes[1].set_aspect('equal')

# 전체 figure 제목
plt.suptitle('이차형식 f(x) = x^TAx의 부호 특성', fontsize=14, fontweight='bold')
# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()


---
## 5. 외적과 행렬의 제곱근

### 📌 외적 (Outer Product)
두 벡터 $x, y$의 외적:
$$x^T y = \text{(행렬)}$$

특히 $x^T x$는 **대칭행렬**이자 **양의 준정부호 행렬**!

### 📌 행렬의 제곱근
$B = \sqrt{A} \Leftrightarrow B^2 = A$이고 $A \geq 0$

**PDM인 경우**: $A = PDP^T$로 대각화되면:
$$\sqrt{A} = P\sqrt{D}P^T$$

In [ ]:
# ── 외적 예제 ───

# 외적 계산 예제
print('외적 (Outer Product) 예제')
# 구분선 출력 — 섹션 구분
print('=' * 50)

x = np.array([1., 2., 3.])
y = np.array([4., 5., 6.])

print('x =', x)
print('y =', y)
print()

# x⊗y — rank-1 행렬
outer_xy = np.outer(x, y)
print('x^T y (외적) =')
print(outer_xy)
print()

outer_xx = np.outer(x, x)
print('x^T x (외적) =')
print(outer_xx)
print('대칭행렬:', np.allclose(outer_xx, outer_xx.T))
# 고윳값·고유벡터 계산 (Av = λv)
ev_xx = np.linalg.eigvalsh(outer_xx)
print('고윳값:', ev_xx.round(4))
print('PSDM (양의 준정부호):', np.all(ev_xx >= -1e-10))


In [ ]:
# ── 행렬 제곱근 ───

# 행렬의 제곱근
print('행렬의 제곱근 계산')
# 구분선 출력 — 섹션 구분
print('=' * 50)

# PDM 행렬
A_pdm2 = np.array([[4., 2.], [2., 3.]])
print('A =')
print(A_pdm2)
print()

# 고윳값 분해로 제곱근 계산
# 고윳값·고유벡터 계산 (Av = λv)
# 대칭행렬 전용 고윳값 분해 (실수 고윳값 보장)
eigenvalues_pdm, P_pdm = np.linalg.eigh(A_pdm2)
print('고윳값:', eigenvalues_pdm)
print()

# sqrt(D)
sqrt_D = np.diag(np.sqrt(eigenvalues_pdm))
print('sqrt(D) =')
print(sqrt_D.round(4))
print()

# sqrt(A) = P sqrt(D) P^T
# √A = P√D P^T (PDM일 때)
sqrt_A = P_pdm @ sqrt_D @ P_pdm.T
print('sqrt(A) = P sqrt(D) P^T =')
print(sqrt_A.round(4))
print()

# 검증: (sqrt_A)^2 = A?
A_reconstructed2 = sqrt_A @ sqrt_A
print('(sqrt(A))^2 =')
print(A_reconstructed2.round(4))
print('원래 A와 일치:', np.allclose(A_reconstructed2, A_pdm2))
print()

# scipy를 이용한 방법과 비교
sqrt_A_scipy = sqrtm(A_pdm2).real
print('scipy.linalg.sqrtm 결과:')
print(sqrt_A_scipy.round(4))
print('두 방법 일치:', np.allclose(sqrt_A, sqrt_A_scipy))


In [ ]:
# ── AA^T = 외적의 합 ───

# 정리 6.16: AA^T = sum of outer products
print('정리 6.16: AA^T = Σ aᵢaᵢᵀ (외적의 합)')
# 구분선 출력 — 섹션 구분
print('=' * 50)

A_col = np.array([[1., 2., 3.], [4., 5., 6.]])
print('A (2×3):')
print(A_col)
print()

# 직접 AA^T 계산
AAT_direct = A_col @ A_col.T
print('AA^T =')
print(AAT_direct)
print()

# 외적의 합으로 계산
AAT_outer = np.zeros((2, 2))
# 반복: 각 원소/조합에 대해 계산·검증
for i in range(3):
    col = A_col[:, i:i+1]
    outer = col @ col.T
# Σ aᵢaᵢ^T
    AAT_outer += outer
    print(f'a{i+1}a{i+1}^T =')
    print(outer)

print('\nΣ aᵢaᵢᵀ =')
print(AAT_outer)
print('AA^T와 일치:', np.allclose(AAT_direct, AAT_outer))


---
## 6. 스펙트럴 클러스터링 (Spectral Clustering)

### 📌 핵심 개념
- **클러스터링**: 데이터를 유사한 그룹으로 묶는 비지도학습
- **스펙트럴 클러스터링**: **Laplacian 행렬의 고유벡터**를 사용하여 클러스터링

### 📌 Laplacian 행렬
$$L = D - W$$
- $W$: 인접행렬 (adjacency matrix) - 가중치 행렬
- $D$: Degree 행렬 - 대각행렬

### 📌 알고리즘
$$\text{argmin}_f \sum w_{ij}(f_i - f_j)^2 = \text{argmin}_f 2[f^T L f]$$

=> **$L$의 고유벡터 = 클러스터 분할 정보를 담고 있음!**

In [ ]:
# ── 5노드 그래프 Laplacian ───

# 강의 예제: 5개 노드 그래프
print('강의 예제: 5개 노드 그래프의 Laplacian 행렬')
# 구분선 출력 — 섹션 구분
print('=' * 60)

# 인접행렬 W (가중치 포함)
W = np.array([
    [5, 4, 4, 0, 0],
    [4, 5, 4, 0, 0],
    [4, 4, 5, 1, 0],
    [0, 0, 1, 5, 4],
    [0, 0, 0, 4, 5]
], dtype=float)

print('인접행렬 W:')
print(W)
print()

# Degree 행렬 D
degrees = W.sum(axis=1)
D_mat = np.diag(degrees)
print('Degree 행렬 D:')
print(D_mat)
print(f'각 노드의 degree: {degrees}')
print()

# Laplacian 행렬
# L = D - W (그래프 Laplacian)
L = D_mat - W
print('Laplacian 행렬 L = D - W:')
print(L)


In [ ]:
# ── Laplacian 고윳값·Fiedler 벡터 ───

# Laplacian의 고윳값과 고유벡터
print('Laplacian 행렬의 고윳값과 고유벡터')
# 구분선 출력 — 섹션 구분
print('=' * 60)

# 고윳값·고유벡터 계산 (Av = λv)
# 대칭행렬 전용 고윳값 분해 (실수 고윳값 보장)
eigenvalues_L, eigenvectors_L = np.linalg.eigh(L)
print('고윳값 (오름차순):')
for i, lam in enumerate(eigenvalues_L):
    print(f'  λ{i+1} = {lam:.4f}')
print()
print('고유벡터 (열벡터):')
print(np.round(eigenvectors_L, 4))
print()

# 두 번째로 작은 고유벡터 (Fiedler vector) - 클러스터링에 사용
# 2번째 최소 고유벡터 → 클러스터
fiedler_vector = eigenvectors_L[:, 1]
print('Fiedler 벡터 (두 번째로 작은 고유벡터):')
print(np.round(fiedler_vector, 4))
print()
print('클러스터 분할:')
cluster_A = np.where(fiedler_vector > 0)[0] + 1
cluster_B = np.where(fiedler_vector <= 0)[0] + 1
print(f'  클러스터 A (양수): 노드 {cluster_A}')
print(f'  클러스터 B (음수/0): 노드 {cluster_B}')


In [ ]:
# ── 스펙트럴 vs K-means ───

# 시각화: 스펙트럴 클러스터링 vs K-means 비교
np.random.seed(42)

# 두 개의 동심원 데이터 생성 (K-means로 분리 불가)
X_circles, y_circles = make_circles(n_samples=300, noise=0.05, factor=0.4, random_state=42)

# 두 개의 반달 형태 데이터 생성
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)

# Matplotlib — 개념을 그림으로 시각화
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

datasets = [(X_circles, y_circles, '동심원 데이터'),
            (X_moons, y_moons, '반달 데이터')]

colors_cluster = ['#e74c3c', '#3498db', '#2ecc71']

for row, (X, y, title) in enumerate(datasets):
    # 원본 데이터
    ax = axes[row, 0]
    ax.scatter(X[:, 0], X[:, 1], c='gray', alpha=0.6, s=20)
    ax.set_title(f'{title}\n(원본)', fontweight='bold')
    ax.set_xlabel('x₁')
    ax.set_ylabel('x₂')
    ax.grid(True, alpha=0.3)
    
    # K-means
    ax = axes[row, 1]
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    kmeans_labels = kmeans.fit_predict(X)
# 반복: 각 원소/조합에 대해 계산·검증
    for k in range(2):
        mask = kmeans_labels == k
        ax.scatter(X[mask, 0], X[mask, 1], c=colors_cluster[k], alpha=0.7, s=20, label=f'클러스터 {k+1}')
    ax.set_title(f'K-means 결과\n(선형 분리 → 실패)', fontweight='bold')
    ax.set_xlabel('x₁')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # 스펙트럴 클러스터링
    ax = axes[row, 2]
# Laplacian 고유벡터로 비선형 분리
    spectral = SpectralClustering(n_clusters=2, affinity='rbf', random_state=42)
    spectral_labels = spectral.fit_predict(X)
# 반복: 각 원소/조합에 대해 계산·검증
    for k in range(2):
        mask = spectral_labels == k
        ax.scatter(X[mask, 0], X[mask, 1], c=colors_cluster[k], alpha=0.7, s=20, label=f'클러스터 {k+1}')
    ax.set_title(f'Spectral Clustering 결과\n(고유벡터 → 성공! ✓)', fontweight='bold')
    ax.set_xlabel('x₁')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

# 전체 figure 제목
plt.suptitle('스펙트럴 클러스터링 vs K-means 비교\n(비선형 구조에서 스펙트럴 클러스터링이 우수!)', 
             fontsize=14, fontweight='bold')
# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()


In [ ]:
# ── 스펙트럴 클러스터링 직접 구현 ───

# 스펙트럴 클러스터링 알고리즘 직접 구현
# W→L→고유벡터→K-means
# 함수 spectral_clustering_manual: 알고리즘·목적 설명은 docstring 참고
def spectral_clustering_manual(X, n_clusters=2, sigma=1.0):
    """
    스펙트럴 클러스터링 직접 구현
    Gaussian Kernel을 사용하여 인접행렬 W를 구성
    """
    n = len(X)
    
    # Step 1: 가우시안 커널로 인접행렬 W 구성
    W = np.zeros((n, n))
# 반복: 각 원소/조합에 대해 계산·검증
    for i in range(n):
# 반복: 각 원소/조합에 대해 계산·검증
        for j in range(n):
            dist_sq = np.sum((X[i] - X[j])**2)
            W[i, j] = np.exp(-dist_sq / (2 * sigma**2))
    np.fill_diagonal(W, 0)  # 자기 자신과의 연결 제거
    
    # Step 2: Degree 행렬 D 구성
    D_vec = W.sum(axis=1)
    D = np.diag(D_vec)
    
    # Step 3: Laplacian 행렬 L = D - W 구성
    L = D - W
    
    # Step 4: L의 고유벡터 계산 (작은 고윳값부터)
# 고윳값·고유벡터 계산 (Av = λv)
# 대칭행렬 전용 고윳값 분해 (실수 고윳값 보장)
    eigenvalues, eigenvectors = np.linalg.eigh(L)
    
    # Step 5: 두 번째부터 n_clusters번째 고유벡터 선택 (첫 번째는 상수벡터)
    features = eigenvectors[:, 1:n_clusters]
    
    # Step 6: 선택한 고유벡터 공간에서 K-means 클러스터링
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(features)
    
    return labels, eigenvalues, eigenvectors

# 작은 예제로 단계별 설명
print('스펙트럴 클러스터링 단계별 설명')
# 구분선 출력 — 섹션 구분
print('=' * 60)

# 간단한 데이터 생성
np.random.seed(42)
X_small = np.vstack([
    np.random.randn(30, 2) * 0.5 + np.array([0, 0]),
    np.random.randn(30, 2) * 0.5 + np.array([4, 0])
])

labels_manual, eigenvalues_manual, eigenvectors_manual = spectral_clustering_manual(X_small, n_clusters=2, sigma=1.0)

print('첫 5개 고윳값:', eigenvalues_manual[:5].round(4))
print('=> 첫 번째 고윳값 ≈ 0 (connected graph의 특성)')
print('=> 두 번째 고윳값이 클러스터 분리 정보 담음 (Fiedler 값)')

# 시각화
# Matplotlib — 개념을 그림으로 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 원본
axes[0].scatter(X_small[:, 0], X_small[:, 1], c='gray', alpha=0.7, s=30)
axes[0].set_title('Step 1: 원본 데이터', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Fiedler 벡터 (두 번째 고유벡터)
fiedler = eigenvectors_manual[:, 1]
colors_fiedler = ['#e74c3c' if f > 0 else '#3498db' for f in fiedler]
axes[1].scatter(range(len(fiedler)), fiedler, c=colors_fiedler, s=30)
axes[1].axhline(y=0, color='k', lw=2, linestyle='--')
axes[1].set_title('Step 2: Fiedler 벡터\n(두 번째 고유벡터)', fontweight='bold')
axes[1].set_xlabel('노드 인덱스')
axes[1].set_ylabel('고유벡터 값')
axes[1].grid(True, alpha=0.3)

# 최종 클러스터링 결과
colors_cluster2 = ['#e74c3c', '#3498db']
# 반복: 각 원소/조합에 대해 계산·검증
for k in range(2):
    mask = labels_manual == k
    axes[2].scatter(X_small[mask, 0], X_small[mask, 1],
                   c=colors_cluster2[k], alpha=0.7, s=30, label=f'클러스터 {k+1}')
axes[2].set_title('Step 3: 클러스터링 결과', fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

# 전체 figure 제목
plt.suptitle('스펙트럴 클러스터링 직접 구현: Laplacian 고유벡터 활용', fontsize=13, fontweight='bold')
# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()


In [ ]:
# ── 가우시안 커널 ───

# 가우시안 커널 시각화
print('가우시안 커널 (Gaussian Kernel)')
# 구분선 출력 — 섹션 구분
print('=' * 50)
print('두 노드 간 가중치:')
print('w_ij = exp(-||x_i - x_j||² / 2σ²)')
print()
print('가까울수록 가중치 크고, 멀수록 가중치 작다!')

# 거리에 따른 가중치 변화
distances = np.linspace(0, 5, 200)

# Matplotlib — 개념을 그림으로 시각화
fig, ax = plt.subplots(figsize=(10, 5))

sigmas = [0.5, 1.0, 2.0]
colors_sigma = ['#e74c3c', '#3498db', '#2ecc71']

for sigma, color in zip(sigmas, colors_sigma):
# w_ij = exp(-d²/2σ²)
    weights = np.exp(-distances**2 / (2 * sigma**2))
    ax.plot(distances, weights, color=color, lw=2.5, label=f'σ = {sigma}')

ax.axhline(y=0, color='k', lw=0.5)
ax.set_xlabel('거리 ||x_i - x_j||', fontsize=13)
ax.set_ylabel('가중치 w_ij', fontsize=13)
ax.set_title('가우시안 커널 함수: w_ij = exp(-d²/2σ²)\n거리가 가까울수록 가중치↑, 멀수록 가중치↓', fontsize=12, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()


---
## 📋 전체 요약

| 개념 | 정의 | 핵심 성질 |
|------|------|----------|
| **고윳값/고유벡터** | $Av = \lambda v$ | 방향 불변, 크기만 $\lambda$배 |
| **특성다항식** | $\det(\lambda I - A) = 0$ | 고윳값 찾는 방정식 |
| **대각화** | $P^{-1}AP = D$ | 독립 고유벡터가 n개이면 가능 |
| **대칭행렬 대각화** | $A = Q^T D Q$ | 항상 가능, 고유벡터들이 직교 |
| **PDM** | $x^T Ax > 0$ | 고윳값 모두 양수 ↔ PDM |
| **행렬의 제곱근** | $\sqrt{A} = P\sqrt{D}P^T$ | PDM에서만 정의 |
| **스펙트럴 클러스터링** | $L = D - W$의 고유벡터 | 비선형 클러스터링 가능 |

In [ ]:
# ── 종합 정리 시각화 ───

# 종합 정리 시각화
# Matplotlib — 개념을 그림으로 시각화
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# 1. 고유벡터의 불변 방향
ax = axes[0]
A_vis = np.array([[2., 1.], [1., 3.]])
# 고윳값·고유벡터 계산 (Av = λv)
# 대칭행렬 전용 고윳값 분해 (실수 고윳값 보장)
eigvals_vis, eigvecs_vis = np.linalg.eigh(A_vis)
theta = np.linspace(0, 2*np.pi, 100)
circle_vis = np.array([np.cos(theta), np.sin(theta)])
transformed_vis = A_vis @ circle_vis
ax.plot(circle_vis[0], circle_vis[1], 'b--', alpha=0.5, lw=1.5)
ax.plot(transformed_vis[0], transformed_vis[1], 'b-', lw=2)
cols = ['red', 'green']
# 반복: 각 원소/조합에 대해 계산·검증
for i in range(2):
    v = eigvecs_vis[:, i]
    ax.annotate('', xy=v*2, xytext=[0,0], arrowprops=dict(arrowstyle='->', color=cols[i], lw=2.5))
    ax.annotate('', xy=A_vis@v, xytext=[0,0], arrowprops=dict(arrowstyle='->', color=cols[i], lw=2, linestyle='dashed', alpha=0.5))
ax.set_xlim(-5,5); ax.set_ylim(-5,5)
ax.axhline(0, color='k', lw=0.5); ax.axvline(0, color='k', lw=0.5)
ax.set_title('① 고유벡터: 방향 불변', fontweight='bold', fontsize=11)
ax.grid(True, alpha=0.3); ax.set_aspect('equal')

# 2. 특성다항식
ax = axes[1]
A_char = np.array([[1., -8.], [1., -5.]])
lam_vis = np.linspace(-5, 2, 300)
f_vis = [(l-1)*(l+5)+8 for l in lam_vis]
ax.plot(lam_vis, f_vis, 'b-', lw=2)
ax.axhline(0, color='k', lw=1)
ax.plot([-3,-1],[0,0],'ro',markersize=10,zorder=5,label='고윳값')
ax.set_xlabel('λ'); ax.set_ylabel('f(λ)')
ax.set_title('② 특성다항식 f(λ)=0의 해', fontweight='bold', fontsize=11)
ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(-5,15)

# 3. 대각화 구조
ax = axes[2]
ax.axis('off')
text = 'A = P D P⁻¹\n\nP: 고유벡터 행렬\nD: 고윳값 대각행렬\n\n대칭행렬:\nA = Q^T D Q\n(Q는 정규직교행렬)'
ax.text(0.5, 0.5, text, ha='center', va='center', fontsize=13,
        transform=ax.transAxes,
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange', lw=2))
ax.set_title('③ 행렬의 대각화', fontweight='bold', fontsize=11)

# 4. PDM 이차형식
ax = axes[3]
x_r = np.linspace(-2, 2, 60)
y_r = np.linspace(-2, 2, 60)
Xm, Ym = np.meshgrid(x_r, y_r)
A_pdm_vis = np.array([[3., 1.], [1., 2.]])
Zm = np.array([[np.array([Xm[i,j],Ym[i,j]])@A_pdm_vis@np.array([Xm[i,j],Ym[i,j]]) 
# 반복: 각 원소/조합에 대해 계산·검증
                for j in range(60)] for i in range(60)])
c = ax.contourf(Xm, Ym, Zm, levels=15, cmap='RdYlGn')
plt.colorbar(c, ax=ax)
ax.set_title('④ 양의 정부호 행렬\nx^TAx > 0', fontweight='bold', fontsize=11)
ax.set_xlabel('x₁'); ax.set_ylabel('x₂')

# 5. 행렬 제곱근
ax = axes[4]
ax.axis('off')
text = ('행렬 제곱근\n\nB = √A  ⟺  B² = A\n\nPDM이면:\n√A = P√D Pᵀ\n\n예시:\nA = [[4,2],[2,3]]\n√A 계산 가능')
ax.text(0.5, 0.5, text, ha='center', va='center', fontsize=12,
        transform=ax.transAxes,
        bbox=dict(boxstyle='round', facecolor='lightcyan', edgecolor='teal', lw=2))
ax.set_title('⑤ 행렬의 제곱근', fontweight='bold', fontsize=11)

# 6. 스펙트럴 클러스터링
ax = axes[5]
# 6개 핵심 개념 한눈에
X_c, y_c = make_circles(n_samples=150, noise=0.05, factor=0.4, random_state=42)
sc = SpectralClustering(n_clusters=2, affinity='rbf', random_state=42)
labels_c = sc.fit_predict(X_c)
colors_c = ['#e74c3c', '#3498db']
# 반복: 각 원소/조합에 대해 계산·검증
for k in range(2):
    mask = labels_c == k
    ax.scatter(X_c[mask,0], X_c[mask,1], c=colors_c[k], alpha=0.7, s=20)
ax.set_title('⑥ 스펙트럴 클러스터링\nL=D-W의 고유벡터 활용', fontweight='bold', fontsize=11)
ax.grid(True, alpha=0.3)

# 전체 figure 제목
plt.suptitle('Chapter 06: 고윳값, 고유벡터, 행렬의 대각화 - 전체 요약', 
             fontsize=14, fontweight='bold', y=1.01)
# 서브플롯 간격 자동 조정
plt.tight_layout()
# 그래프 화면 출력
plt.show()

print('\n학습 완료! 🎉')
